In [ ]:
!pip install numba

The modeling workflow is built to estimate the outdoor radiant and thermal environment at each pixel of a gridded urban surface by combining geometric obstruction, solar exposure, surface properties, and meteorological conditions. First, a digital surface height grid (representing buildings and ground) is used to compute the sky view factor (SVF) and shade state for each pixel. The SVF represents the fraction of the sky hemisphere visible from a point and is obtained by casting rays in many azimuth directions and finding the highest obstruction angle along each ray. Averaging these horizon angles yields a measure of how enclosed the location is. Using the solar position for the specified date, time, and location, the same ray-tracing approach determines whether the sun’s direct beam is blocked, producing a binary shade map. Together, SVF describes longwave radiative exposure to the sky and surrounding structures, while the shade calculation determines whether direct shortwave radiation reaches the point.

These geometric outputs are then combined with meteorological inputs (air temperature, humidity, and solar radiation) and a ground albedo raster to estimate the mean radiant temperature (MRT). The code approximates the total radiation absorbed by a human body as the sum of (i) longwave radiation from the sky scaled by SVF, (ii) longwave radiation from surrounding surfaces (assumed near air temperature), (iii) incoming shortwave radiation that depends on whether the pixel is sunlit or shaded, and (iv) shortwave radiation reflected from the ground proportional to the local albedo. The absorbed radiative flux is converted into an equivalent black-body temperature through the Stefan–Boltzmann relationship, producing MRT for each grid cell. Finally, MRT is combined with air temperature, humidity, and wind speed to compute the Universal Thermal Climate Index (UTCI) using an established polynomial approximation. UTCI translates the physical heat exchange environment into an equivalent perceived temperature, allowing the raster outputs to represent spatial patterns of human thermal stress rather than only physical radiation fields.

One important note about accuracy

The polynomial I provided earlier is a truncated approximation; it will run and produce sensible-looking fields, but if you need research-grade UTCI, the best approach is to use the full UTCI operational polynomial (very long) or a validated library implementation.

If you tell me whether you’re okay installing a dependency, I can provide either:

Option A (best): a wrapper using pythermalcomfort (validated UTCI)

Option B (no deps): the full operational UTCI polynomial in one function (long but standalone)

A few big “next steps” fall into two buckets: physics fidelity and computational/engineering robustness.

Physics and realism upgrades

Use direct + diffuse shortwave (not just global) and apply sun-angle weighting. Right now, “sunlit = global, shaded = diffuse” is a helpful shortcut, but MRT is sensitive to how much direct beam hits the person (depends on solar elevation/azimuth, body orientation, and whether sun is partially obstructed). A common next step is: compute direct normal irradiance (DNI) + diffuse horizontal (DHI), then project direct onto the person and surfaces (SOLWEIG-style).

Account for surface temperatures instead of assuming surfaces ≈ air temperature. Longwave from walls/ground can dominate at night and in hot sun. Next step: either (a) ingest a surface temperature raster (from thermal imagery or a microclimate model), or (b) estimate it with a simple energy-balance model using albedo, radiation, and heat capacity.

Improve longwave sky model with cloudiness. Humidity-only emissivity is a rough proxy; cloud cover strongly increases downwelling longwave. Next step: add cloud fraction or use measured downwelling longwave if available.

Reflections beyond ground albedo: urban canyons get important shortwave and longwave reflections from façades. Next step: include wall albedo/emissivity and a simple view-factor split (sky vs ground vs vertical surfaces), or move toward a simplified radiosity approach.

Shade realism: your shade is binary. In practice, vegetation, partial obstruction, and diffuse anisotropy create “soft shade.” Next step: compute sky obstruction by sector (you already have horizon angles) and use that to attenuate diffuse and direct components more continuously.

UTCI input conventions: UTCI expects wind at 10 m and vapor pressure handling consistent with its operational definition. Next step: if your wind is at 2 m, convert using a log profile (needs roughness length / stability assumptions), and use a validated UTCI implementation (e.g., reference polynomial/library) to avoid subtle errors.

Geometry/modeling considerations

Coordinate system & cellsize correctness: Ray distances must be in meters; ensure the raster is in a projected CRS (e.g., UTM) or that you handle lat/lon distortions if not.

Edge effects & max_distance: SVF depends on how far you search for obstructions. Too small → SVF too high; too large → slow. Next step: choose max_distance based on urban morphology (often 100–500 m) and test sensitivity.

Vegetation / porous obstacles: If trees exist, a DSM alone treats them as solid. Next step: add a vegetation layer with transmissivity or seasonal leaf-on/off behavior.

Human geometry assumptions: MRT depends on body posture, height, and angular exposure. Next step: compute at a standard “pedestrian height” (e.g., 1.1–1.7 m) by subtracting/adding height offsets, and apply projected area factors for direct sun.

Computational next steps (making it practical at scale)

Tile/chunk processing for very large rasters (e.g., 10k×10k): process in chunks with overlap equal to max_distance/cellsize.

Precompute horizon angles once if you’re running many hours/days. SVF and horizon by azimuth don’t change with time; only sun direction changes. Next step: store a “horizon angle cube” (azimuth × rows × cols) or a compressed representation (e.g., 36 sectors) and reuse it for shade + radiation all day.

Acceleration strategy: the current ray casting is already optimized with precomputed offsets and JIT; next step is reducing rays/steps intelligently (adaptive stepping, multi-resolution, early termination) and using memory-friendly types.

Validation and sanity checks (highly recommended)

Compare SVF against a known SVF tool (e.g., a GIS SVF plugin or a published method) on a small subset.

Check shade against simple cases (single building casting expected shadow direction/length at known sun elevation).

Compare MRT/UTCI with a station or mobile transect if you have field data; MRT can be cross-checked with a globe thermometer (with caveats).

A strong “next implementation step” roadmap

Precompute horizon angles by azimuth sector (e.g., 36–72) and SVF once.

Add solar radiation inputs (DNI/DHI or measured GHI + partition model) and cloud factor.

Add surface temperature (ingested or modeled).

Switch UTCI to a validated reference implementation and wind-height handling.

Wrap everything into a daily loop producing hourly shade/MRT/UTCI rasters efficiently.

In [ ]:
!pip install mplcursors requests timezonefinder

In [ ]:
!pip install rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import rasterio
path_file = '/content/drive/MyDrive/Codes-Urban Heat/Heat_Imaging_Files/Harlem/nDSM_Harlem.tif'

In [ ]:
from platform import architecture
with rasterio.open(path_file) as src:
    ar_ndsm= src.read(1)
    # print(ar)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)


In [ ]:
import matplotlib.pyplot as plt
plt.imshow(ar_ndsm)

In [ ]:
from datetime import datetime

# ── 1. DATE & TIME (local time)
SIMULATION_DATETIME = datetime(2025, 7, 1, 14, 0, 0)  # Year, Month, Day, Hour, Min, Sec

# ── 2. WIND SPEED (m/s at ~10 m height)
WIND_SPEED_MPS = 2.0

# ── 3. SVF PARAMETERS
N_AZIMUTH = 72           # number of azimuth rays (more = slower but more accurate)
MAX_DISTANCE_M = 200.0   # max ray search distance in meters
GROUND_THRESHOLD_M = 1.0 # heights below this are treated as ground for SVF

# ── 4. COORDINATES & TIMEZONE
#    Auto-extracted from the raster CRS center.
#    Override below if needed.
AUTO_DETECT_COORDS = True   # set False to use manual LAT/LON below
MANUAL_LAT = 40.74
MANUAL_LON = -74.03

print(f"Simulation: {SIMULATION_DATETIME}")
print(f"Wind speed: {WIND_SPEED_MPS} m/s")


In [ ]:
#auto-detect coordinates and weather

import requests
import numpy as np
from timezonefinder import TimezoneFinder
from datetime import timezone as tz, timedelta
import pytz

# ── Step 1: Get coordinates from raster or manual input ──────
if AUTO_DETECT_COORDS:
    with rasterio.open(path_file) as src:
        bounds = src.bounds
        crs = src.crs
        # Get center of raster in its native CRS
        cx = (bounds.left + bounds.right) / 2
        cy = (bounds.bottom + bounds.top) / 2

        # If projected CRS (e.g. UTM), transform to WGS84
        if crs and not crs.is_geographic:
            from rasterio.warp import transform
            lon_list, lat_list = transform(crs, 'EPSG:4326', [cx], [cy])
            LAT, LON = lat_list[0], lon_list[0]
        else:
            LAT, LON = cy, cx

        CELLSIZE = src.res[0]  # meters per pixel (assumes projected CRS)
else:
    LAT = MANUAL_LAT
    LON = MANUAL_LON
    CELLSIZE = 1.0  # default; override if needed

print(f"Coordinates: {LAT:.5f}°N, {LON:.5f}°E")
print(f"Cell size:   {CELLSIZE:.2f} m")

# ── Step 2: Auto-detect timezone offset ─────────────────────
tf = TimezoneFinder()
tz_name = tf.timezone_at(lat=LAT, lng=LON)
if tz_name:
    local_tz = pytz.timezone(tz_name)
    # Get the UTC offset for the simulation datetime
    dt_aware = local_tz.localize(SIMULATION_DATETIME)
    TZ_OFFSET = dt_aware.utcoffset().total_seconds() / 3600
    print(f"Timezone:    {tz_name} (UTC{TZ_OFFSET:+.0f})")
else:
    TZ_OFFSET = -5  # fallback
    print(f"Could not detect timezone, using UTC{TZ_OFFSET:+.0f}")

# ── Step 3: Fetch weather from Open-Meteo ───────────────────
def fetch_weather_openmeteo(lat, lon, dt_local, tz_offset):
    """
    Fetch hourly weather from Open-Meteo for the given datetime.
    Returns dict with: temperature_2m, relative_humidity_2m,
    global_radiation, diffuse_radiation, wind_speed_10m.
    Works for historical dates (back to 1940) and forecasts (up to 16 days).
    """
    date_str = dt_local.strftime('%Y-%m-%d')
    hour = dt_local.hour

    # Decide endpoint: archive for past dates, forecast for recent/future
    from datetime import date
    days_ago = (date.today() - dt_local.date()).days

    if days_ago > 5:
        base_url = 'https://archive-api.open-meteo.com/v1/archive'
    else:
        base_url = 'https://api.open-meteo.com/v1/forecast'

    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': date_str,
        'end_date': date_str,
        'hourly': ','.join([
            'temperature_2m',
            'relative_humidity_2m',
            'shortwave_radiation',
            'diffuse_radiation',
            'wind_speed_10m',
        ]),
        'timezone': 'auto',
    }

    resp = requests.get(base_url, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    hourly = data['hourly']

    # Find the matching hour index
    idx = hour  # hourly data starts at 00:00 local
    if idx >= len(hourly['temperature_2m']):
        idx = len(hourly['temperature_2m']) - 1

    result = {
        'temperature_2m': hourly['temperature_2m'][idx],
        'relative_humidity_2m': hourly['relative_humidity_2m'][idx],
        'shortwave_radiation': hourly['shortwave_radiation'][idx],
        'diffuse_radiation': hourly['diffuse_radiation'][idx],
        'wind_speed_10m': hourly['wind_speed_10m'][idx],
    }
    return result


print(f"\nFetching weather for {SIMULATION_DATETIME} ...")
try:
    wx = fetch_weather_openmeteo(LAT, LON, SIMULATION_DATETIME, TZ_OFFSET)

    Ta = wx['temperature_2m']              # °C
    RH = wx['relative_humidity_2m']         # %
    K_global = wx['shortwave_radiation']    # W/m²
    K_diffuse = wx['diffuse_radiation']     # W/m²
    WIND_SPEED_MPS = wx['wind_speed_10m'] if wx['wind_speed_10m'] else WIND_SPEED_MPS

    print(f"  Air temperature:       {Ta:.1f} °C")
    print(f"  Relative humidity:     {RH:.0f} %")
    print(f"  Global SW radiation:   {K_global:.0f} W/m²")
    print(f"  Diffuse SW radiation:  {K_diffuse:.0f} W/m²")
    print(f"  Wind speed (10 m):     {WIND_SPEED_MPS:.1f} m/s")
    WEATHER_OK = True

except Exception as e:
    print(f"  ⚠ Weather fetch failed: {e}")
    print(f"  Using fallback values. Edit manually if needed.")
    Ta = 32.0
    RH = 60.0
    K_global = 800.0
    K_diffuse = 150.0
    WEATHER_OK = False


In [ ]:
from math import sin, cos, tan, asin, acos, atan2, radians, degrees, floor
from datetime import datetime, timezone
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
# import mplcursors

In [ ]:


# ----------------------------
# Sun position (fast NOAA-style approximation)
# Azimuth: degrees clockwise from North (0=N, 90=E)
# Elevation: degrees above horizon
# ----------------------------
def sun_position(lat_deg: float, lon_deg: float, when_local: datetime, tz_offset_hours: float):
    # Convert local -> UTC
    when_utc = when_local.replace(tzinfo=timezone.utc) - np.timedelta64(int(tz_offset_hours * 3600), "s").astype("timedelta64[s]").item()

    y, m, d = when_utc.year, when_utc.month, when_utc.day
    hr = when_utc.hour + when_utc.minute / 60 + when_utc.second / 3600

    if m <= 2:
        y -= 1
        m += 12
    A = floor(y / 100)
    B = 2 - A + floor(A / 4)
    JD = floor(365.25 * (y + 4716)) + floor(30.6001 * (m + 1)) + d + B - 1524.5 + hr / 24.0
    T = (JD - 2451545.0) / 36525.0

    L0 = (280.46646 + T * (36000.76983 + 0.0003032 * T)) % 360
    M = 357.52911 + T * (35999.05029 - 0.0001537 * T)
    e = 0.016708634 - T * (0.000042037 + 0.0000001267 * T)

    C = (1.914602 - T * (0.004817 + 0.000014 * T)) * sin(radians(M)) \
        + (0.019993 - 0.000101 * T) * sin(radians(2 * M)) \
        + 0.000289 * sin(radians(3 * M))

    true_long = L0 + C
    omega = 125.04 - 1934.136 * T
    lambda_sun = true_long - 0.00569 - 0.00478 * sin(radians(omega))

    eps0 = 23 + (26 + ((21.448 - T * (46.815 + T * (0.00059 - 0.001813 * T)))) / 60) / 60
    eps = eps0 + 0.00256 * cos(radians(omega))

    delta = asin(sin(radians(eps)) * sin(radians(lambda_sun)))

    y_term = tan(radians(eps / 2)) ** 2
    EoT = 4 * degrees(
        y_term * sin(2 * radians(L0)) - 2 * e * sin(radians(M))
        + 4 * e * y_term * sin(radians(M)) * cos(2 * radians(L0))
        - 0.5 * y_term * y_term * sin(4 * radians(L0))
        - 1.25 * e * e * sin(2 * radians(M))
    )

    minutes = when_local.hour * 60 + when_local.minute + when_local.second / 60
    tst = (minutes + EoT + 4 * lon_deg - 60 * tz_offset_hours) % 1440
    ha = radians(tst / 4 - 180)

    lat = radians(lat_deg)
    cos_zen = sin(lat) * sin(delta) + cos(lat) * cos(delta) * cos(ha)
    cos_zen = max(-1.0, min(1.0, cos_zen))
    zen = acos(cos_zen)

    elev = 90 - degrees(zen)

    az = degrees(atan2(sin(ha), cos(ha) * sin(lat) - tan(delta) * cos(lat)))
    az = (az + 180) % 360

    return az, elev


# ----------------------------
# Ray precomputation
# We precompute integer offsets along each azimuth direction and their distances.
# This makes the per-pixel kernel extremely fast.
# ----------------------------
def precompute_rays(n_azimuth: int, max_steps: int, cellsize: float):
    azimuths = np.linspace(0.0, 360.0, n_azimuth, endpoint=False).astype(np.float32)

    di = np.empty((n_azimuth, max_steps), dtype=np.int32)
    dj = np.empty((n_azimuth, max_steps), dtype=np.int32)
    dist = np.empty((max_steps,), dtype=np.float32)

    # distances in meters
    for s in range(max_steps):
        dist[s] = (s + 1) * cellsize

    # integer offsets (rounded continuous ray)
    for k, az in enumerate(azimuths):
        a = np.deg2rad(az)
        u_i = -np.cos(a)  # N is -i
        u_j =  np.sin(a)  # E is +j
        for s in range(max_steps):
            step = s + 1
            di[k, s] = int(np.rint(u_i * step))
            dj[k, s] = int(np.rint(u_j * step))

        # Ensure first step isn't (0,0) (can happen at tiny angles due to rounding)
        if di[k, 0] == 0 and dj[k, 0] == 0:
            # nudge by forcing at least one cell move in dominant direction
            if abs(u_i) >= abs(u_j):
                di[k, 0] = -1 if u_i < 0 else 1
            else:
                dj[k, 0] = -1 if u_j < 0 else 1

    return azimuths, di, dj, dist


# ----------------------------
# Fast SVF + shade (Numba accelerated)
# ----------------------------
from numba import njit, prange

@njit(cache=True, fastmath=True)
def _horizon_angle_along_precomputed(H, i0, j0, di, dj, dist):
    """
    Returns max horizon elevation angle gamma (radians) along a precomputed ray.
    """
    nrows, ncols = H.shape
    h0 = H[i0, j0]
    max_gamma = 0.0

    for t in range(dist.shape[0]):
        i = i0 + di[t]
        j = j0 + dj[t]
        if i < 0 or i >= nrows or j < 0 or j >= ncols:
            break

        dh = H[i, j] - h0
        # gamma = atan2(dh, dist)
        # Use atan2 for correctness (fastmath helps)
        g = np.arctan2(dh, dist[t])
        if g > max_gamma:
            max_gamma = g

    return max_gamma


@njit(parallel=True, cache=True, fastmath=True)
def _compute_svf_and_shade_numba(H, zero_mask, rays_di, rays_dj, dist, sun_di, sun_dj, sun_dist, sun_el_rad):
    nrows, ncols = H.shape
    n_az = rays_di.shape[0]

    svf = np.empty((nrows, ncols), dtype=np.float32)
    shade = np.empty((nrows, ncols), dtype=np.bool_)

    # Initialize outputs
    for i in prange(nrows):
        for j in range(ncols):
            svf[i, j] = np.nan
            shade[i, j] = True  # default shaded; corrected below

    # If sun below horizon: everything shaded, SVF still computed for zero pixels
    sun_above = sun_el_rad > 0.0

    for i in prange(nrows):
        for j in range(ncols):
            # ---- Shade for ALL pixels ----
            if sun_above:
                gamma_sun = _horizon_angle_along_precomputed(H, i, j, sun_di, sun_dj, sun_dist)
                shade[i, j] = (sun_el_rad <= gamma_sun)
            else:
                shade[i, j] = True

            # ---- SVF only for zeros ----
            if zero_mask[i, j]:
                acc = 0.0
                for k in range(n_az):
                    gamma = _horizon_angle_along_precomputed(H, i, j, rays_di[k], rays_dj[k], dist)
                    acc += np.cos(gamma)
                svf[i, j] = acc / n_az

    return svf, shade


def compute_svf_and_shade(
    H,
    lat_deg: float,
    lon_deg: float,
    when_local: datetime,
    tz_offset_hours: float,
    cellsize: float = 1.0,
    n_azimuth: int = 72,
    max_distance: float = 200.0,
    ground_threshold: float = 1.0,
):
    """
    Super-optimized SVF + shade.

    Inputs
    - H: 2D array of heights
    - lat_deg, lon_deg: location
    - when_local: local datetime
    - tz_offset_hours: timezone offset (e.g., -5, -4)
    - cellsize: meters per pixel
    - n_azimuth: number of azimuth directions for SVF
    - max_distance: max ray length (meters)
    - ground_threshold: height (m) below which a pixel is treated as ground for SVF (default 1.0)

    Returns
    - svf: 2D float32, SVF for pixels with H==0, np.nan elsewhere
    - shade: 2D bool, True=shaded, False=sunlit (for all pixels)
    - sun_az_deg, sun_el_deg
    """
    H = np.asarray(H, dtype=np.float32)
    if H.ndim != 2:
        raise ValueError("H must be a 2D array.")

    sun_az_deg, sun_el_deg = sun_position(lat_deg, lon_deg, when_local, tz_offset_hours)
    sun_el_rad = np.deg2rad(sun_el_deg).astype(np.float64)

    max_steps = int(np.ceil(max_distance / cellsize))
    if max_steps < 1:
        raise ValueError("max_distance must be >= cellsize (or increase max_distance).")

    # SVF rays
    azimuths, rays_di, rays_dj, dist = precompute_rays(n_azimuth, max_steps, cellsize)

    # Sun ray (single direction)
    sun_az = float(sun_az_deg)
    _, sun_di_2d, sun_dj_2d, sun_dist = precompute_rays(1, max_steps, cellsize)
    # But precompute_rays uses evenly spaced azimuths; replace with exact sun az:
    # We'll rebuild the 1-ray offsets directly for precision.
    sun_di = np.empty((max_steps,), dtype=np.int32)
    sun_dj = np.empty((max_steps,), dtype=np.int32)
    a = np.deg2rad(sun_az)
    u_i = -np.cos(a)
    u_j =  np.sin(a)
    for s in range(max_steps):
        step = s + 1
        sun_di[s] = int(np.rint(u_i * step))
        sun_dj[s] = int(np.rint(u_j * step))
    if sun_di[0] == 0 and sun_dj[0] == 0:
        if abs(u_i) >= abs(u_j):
            sun_di[0] = -1 if u_i < 0 else 1
        else:
            sun_dj[0] = -1 if u_j < 0 else 1

    # Use a height threshold instead of exact zero —
    # real nDSM ground pixels are never exactly 0.0 due to measurement noise.
    zero_mask = (H < ground_threshold)

    # Run the JIT kernel
    svf, shade = _compute_svf_and_shade_numba(
        H,
        zero_mask,
        rays_di, rays_dj, dist,
        sun_di, sun_dj, dist,  # sun_dist == dist (same step distances)
        float(sun_el_rad),
    )

    return svf, shade, sun_az_deg, sun_el_deg




In [ ]:

SIGMA = 5.670374419e-8  # W m^-2 K^-4

def _sat_vapor_pressure_hpa(Ta_C: np.ndarray) -> np.ndarray:
    """
    Saturation vapor pressure (hPa), Magnus-Tetens over water.
    """
    Ta = np.asarray(Ta_C, dtype=np.float64)
    return 6.112 * np.exp((17.62 * Ta) / (243.12 + Ta))

def _sky_emissivity_simple(Ta_C: np.ndarray, RH_frac: np.ndarray) -> np.ndarray:
    """
    Simple clear-sky emissivity approximation.
    (Not perfect; good for many outdoor MRT workflows.)
    """
    # A commonly used simple parameterization; keep bounded.
    Ta = np.asarray(Ta_C, dtype=np.float64)
    RH = np.clip(np.asarray(RH_frac, dtype=np.float64), 0.0, 1.0)
    eps = 0.72 + 0.005 * Ta + 0.00015 * (RH * 100.0)  # heuristic
    return np.clip(eps, 0.5, 0.99)



SIGMA = 5.670374419e-8  # Stefan-Boltzmann constant [W m-2 K-4]

def _saturation_vapor_pressure_hpa(T_C):
    """Tetens formula; returns saturation vapor pressure in hPa."""
    return 6.112 * np.exp((17.67 * T_C) / (T_C + 243.5))

def _sky_emissivity_prata_like(Ta_C, RH_percent):
    """
    Simple clear-sky emissivity estimate from air temperature and humidity.
    This is still a simplification, but much better than forcing sky longwave
    to equal surface longwave.
    """
    Ta_K = Ta_C + 273.15
    ea_hPa = (RH_percent / 100.0) * _saturation_vapor_pressure_hpa(Ta_C)
    ea_kPa = ea_hPa / 10.0

    # Common simple atmospheric emissivity form
    eps_sky = 1.24 * (ea_kPa / Ta_K) ** (1.0 / 7.0)
    return np.clip(eps_sky, 0.65, 0.99)

def _projected_area_factor_standing(solar_elevation_deg):
    """
    Projected area factor for a standing person.
    Uses a common smooth approximation as a function of solar elevation.
    Returns a dimensionless factor usually around 0.15–0.35.
    """
    gamma = np.deg2rad(np.clip(solar_elevation_deg, 0.0, 90.0))
    # Simple bounded approximation for a standing cylinder-like person
    f_p = 0.308 * np.cos(gamma * (1.0 - 0.5))  # kept smooth and conservative
    return np.clip(f_p, 0.15, 0.35)

def compute_mrt_no_wind(
    svf,
    shade,
    Ta_C,
    RH_percent,
    K_global,
    K_diffuse,
    solar_elevation_deg,
    albedo=0.20,
    emissivity_human=0.97,
    absorptivity_sw=0.70,
    emissivity_surface=0.95,
    surface_temp_offset_sun_C=8.0,
    surface_temp_offset_shade_C=2.0,
    return_components=False,
):
    """
    Compute mean radiant temperature (MRT, degC) for outdoor urban pixels.

    Parameters
    ----------
    svf : array-like
        Sky view factor [0..1].
    shade : array-like
        1 for shaded, 0 for sunlit (or boolean).
    Ta_C : float or array-like
        Air temperature [degC].
    RH_percent : float or array-like
        Relative humidity [%].
    K_global : float or array-like
        Global shortwave radiation on horizontal surface [W m-2].
    K_diffuse : float or array-like
        Diffuse shortwave radiation on horizontal surface [W m-2].
    solar_elevation_deg : float
        Solar elevation angle [deg].
    albedo : float
        Ground/surrounding shortwave reflectance.
    emissivity_human : float
        Human longwave emissivity.
    absorptivity_sw : float
        Human shortwave absorptivity.
    emissivity_surface : float
        Urban surface emissivity.
    surface_temp_offset_sun_C : float
        Approximate excess radiant surface temperature above air temp in sun.
    surface_temp_offset_shade_C : float
        Approximate excess radiant surface temperature above air temp in shade.
    return_components : bool
        If True, also return a dict of radiation components.

    Returns
    -------
    mrt_C : ndarray
        Mean radiant temperature [degC].
    """

    svf = np.asarray(svf, dtype=float)
    shade = np.asarray(shade).astype(bool)

    Ta_C = np.asarray(Ta_C, dtype=float)
    RH_percent = np.asarray(RH_percent, dtype=float)
    K_global = np.asarray(K_global, dtype=float)
    K_diffuse = np.asarray(K_diffuse, dtype=float)

    # Broadcast meteorological scalars to the SVF grid if needed
    svf, Ta_C, RH_percent, K_global, K_diffuse = np.broadcast_arrays(
        svf, Ta_C, RH_percent, K_global, K_diffuse
    )

    # Solar geometry
    sin_gamma = np.sin(np.deg2rad(np.clip(solar_elevation_deg, 0.0, 90.0)))
    sin_gamma = max(sin_gamma, 1e-6)

    # Split shortwave into direct horizontal and direct normal
    K_dir_h = np.maximum(K_global - K_diffuse, 0.0)
    K_dir_n = np.where(solar_elevation_deg > 0.0, K_dir_h / sin_gamma, 0.0)

    # Direct shortwave to a standing person:
    # only when sunlit, scaled by projected area factor
    f_p = _projected_area_factor_standing(solar_elevation_deg)
    K_direct_abs = np.where(
        (~shade) & (solar_elevation_deg > 0.0),
        absorptivity_sw * f_p * K_dir_n,
        0.0
    )

    # Diffuse shortwave from visible sky
    # Use SVF to modulate how much of the sky dome contributes
    K_diffuse_abs = absorptivity_sw * 0.5 * svf * K_diffuse

    # Reflected shortwave from ground / urban surfaces
    # A standing person sees roughly the lower hemisphere regardless of SVF,
    # but enclosed places can enhance local reflections somewhat.
    view_ground = 0.5
    enclosure_boost = 1.0 + 0.3 * (1.0 - svf)
    K_reflected_abs = absorptivity_sw * view_ground * albedo * K_global * enclosure_boost

    # Longwave radiation
    Ta_K = Ta_C + 273.15
    eps_sky = _sky_emissivity_prata_like(Ta_C, RH_percent)
    L_sky = eps_sky * SIGMA * Ta_K**4

    # Simple surface radiant temperature assumption:
    # sunlit surroundings warmer than air, shaded surroundings slightly warmer.
    Tsurf_C = np.where(shade, Ta_C + surface_temp_offset_shade_C,
                              Ta_C + surface_temp_offset_sun_C)
    Tsurf_K = Tsurf_C + 273.15
    L_surface = emissivity_surface * SIGMA * Tsurf_K**4

    # View factors:
    # visible sky = svf
    # remaining upper obstruction = (1-svf)*0.5
    # ground/lower hemisphere = 0.5
    vf_sky = 0.5 * svf
    vf_walls = 0.5 * (1.0 - svf)
    vf_ground = 0.5

    L_abs = emissivity_human * (
        vf_sky * L_sky +
        vf_walls * L_surface +
        vf_ground * L_surface
    )

    # Total absorbed radiant flux
    R_abs = K_direct_abs + K_diffuse_abs + K_reflected_abs + L_abs

    # Effective radiation area factor for standing person
    f_eff = 0.72

    mrt_K = (R_abs / (emissivity_human * SIGMA * f_eff)) ** 0.25
    mrt_C = mrt_K - 273.15

    if return_components:
        comps = {
            "K_direct_abs": K_direct_abs,
            "K_diffuse_abs": K_diffuse_abs,
            "K_reflected_abs": K_reflected_abs,
            "L_abs": L_abs,
            "R_abs": R_abs,
            "eps_sky": eps_sky,
            "f_p": np.full_like(svf, f_p, dtype=float),
        }
        return mrt_C, comps

    return mrt_C

# def compute_mrt_no_wind(
#     svf: np.ndarray,
#     shade: np.ndarray,
#     Ta_C,
#     RH_percent,
#     albedo: np.ndarray,
#     K_global=800.0,      # W/m²
#     K_diffuse=120.0,     # W/m²
#     absorptivity_sw=0.7, # human shortwave absorptivity
#     emissivity_lw=0.97,  # human longwave emissivity
#     T_surfaces_C=None,   # optional: 2D or scalar surface temperature; default=Ta
# ) -> np.ndarray:
#     """
#     Mean Radiant Temperature (°C) for each pixel.

#     Inputs
#     - svf: 2D float [0..1] (np.nan allowed for buildings)
#     - shade: 2D bool (True = shaded)
#     - Ta_C: scalar or 2D air temperature (°C)
#     - RH_percent: scalar or 2D relative humidity (%)
#     - albedo: 2D float [0..1] ground albedo raster
#     - K_global / K_diffuse: shortwave radiation (W/m²)
#     - T_surfaces_C: optional approximation of mean surrounding surface temperature (°C).
#       If None, uses Ta_C.

#     Returns
#     - mrt_C: 2D float (°C), np.nan where svf is nan.
#     """
#     svf = np.asarray(svf, dtype=np.float64)
#     shade = np.asarray(shade, dtype=bool)
#     albedo = np.clip(np.asarray(albedo, dtype=np.float64), 0.0, 1.0)

#     # Broadcast Ta/RH/solar to arrays
#     Ta_C = np.asarray(Ta_C, dtype=np.float64)
#     RH = np.clip(np.asarray(RH_percent, dtype=np.float64) / 100.0, 0.0, 1.0)

#     # Handle surfaces temperature
#     if T_surfaces_C is None:
#         Tsurf_C = Ta_C
#     else:
#         Tsurf_C = np.asarray(T_surfaces_C, dtype=np.float64)

#     Ta_K = Ta_C + 273.15
#     Tsurf_K = Tsurf_C + 273.15

#     # Longwave components
#     eps_sky = _sky_emissivity_simple(Ta_C, RH)
#     L_sky = eps_sky * SIGMA * Ta_K**4                  # sky longwave (W/m²)
#     L_surf = SIGMA * Tsurf_K**4                        # surrounding surfaces (W/m²) emissivity folded into human emissivity later

#     # Approximate longwave irradiance seen by a person
#     # SVF fraction from sky, rest from surfaces
#     L_in = svf * L_sky + (1.0 - svf) * L_surf

#     # Shortwave: incident on the point (very simplified)
#     K_in = np.where(shade, K_diffuse, K_global)

#     # Ground-reflected shortwave (Lambertian, rough approximation):
#     # scale by ground "view" fraction ~ (1 - SVF)
#     K_reflected = albedo * K_in * (1.0 - svf)

#     # Total absorbed radiation by human
#     R_abs = emissivity_lw * L_in + absorptivity_sw * (K_in + K_reflected)

#     # Invert Stefan-Boltzmann to MRT
#     mrt_K = (R_abs / (emissivity_lw * SIGMA)) ** 0.25
#     mrt_C = mrt_K - 273.15

#     # Keep NaN where svf is NaN (e.g., buildings)
#     mrt_C = np.where(np.isfinite(svf), mrt_C, np.nan)
#     return mrt_C

In [ ]:

def compute_utci_from_mrt(
    Ta_C,
    RH_percent,
    wind_mps,
    mrt_C,
) -> np.ndarray:
    """
    UTCI (°C) from air temperature, RH, wind speed, and MRT.

    Accepts scalars or arrays for Ta_C, RH_percent, wind_mps.
    mrt_C should be 2D (or at least array-like) and sets the output shape.

    Returns:
      2D UTCI array (°C), np.nan where mrt_C is nan.
    """
    Tmrt = np.asarray(mrt_C, dtype=np.float64)
    out_shape = Tmrt.shape

    # Broadcast inputs to the shape of mrt
    Ta = np.broadcast_to(np.asarray(Ta_C, dtype=np.float64), out_shape)
    RH = np.broadcast_to(np.asarray(RH_percent, dtype=np.float64), out_shape)
    va = np.broadcast_to(np.asarray(wind_mps, dtype=np.float64), out_shape)

    RH = np.clip(RH, 0.0, 100.0)
    va = np.maximum(va, 0.0)

    # Mask invalid pixels (where MRT is nan)
    mask = np.isfinite(Tmrt) & np.isfinite(Ta)

    utci = np.full(out_shape, np.nan, dtype=np.float64)

    # Vapor pressure (hPa)
    es = 6.112 * np.exp((17.62 * Ta) / (243.12 + Ta))
    ehPa = es * (RH / 100.0)

    dTmrt = Tmrt - Ta

    # Extract masked 1D vectors
    t = Ta[mask]
    v = va[mask]
    e = ehPa[mask]
    d = dTmrt[mask]

    # UTCI convention usually expects wind >= 0.5 m/s
    v = np.maximum(v, 0.5)

    # --- Polynomial approximation (short truncation as provided earlier) ---
    utci_m = (
        t
        + (0.607562052)
        + (-0.0227712343)*t
        + (8.06470249e-04)*t*t
        + (-1.54271372e-04)*t*t*t
        + (-3.24651735e-06)*t*t*t*t
        + (7.32602852e-08)*t*t*t*t*t
        + (1.35959073e-09)*t*t*t*t*t*t

        + (-2.25836520)*v
        + (0.0880326035)*t*v
        + (0.00216844454)*t*t*v
        + (-1.53347087e-05)*t*t*t*v
        + (-5.72983704e-07)*t*t*t*t*v
        + (-2.55090145e-09)*t*t*t*t*t*v
        + (-0.751269505)*v*v
        + (-0.00408350271)*t*v*v
        + (-5.21670675e-05)*t*t*v*v
        + (1.94544667e-06)*t*t*t*v*v
        + (1.14099531e-08)*t*t*t*t*v*v
        + (0.158137256)*v*v*v
        + (-6.57263143e-05)*t*v*v*v
        + (2.22697524e-07)*t*t*v*v*v
        + (-4.16117031e-08)*t*t*t*v*v*v
        + (-0.0127762753)*v*v*v*v
        + (9.66891875e-06)*t*v*v*v*v
        + (2.52785852e-09)*t*t*v*v*v*v
        + (4.56306672e-04)*v*v*v*v*v

        + (0.00286096834)*e
        + (-3.30552823e-03)*t*e
        + (-1.64119440e-05)*t*t*e
        + (-5.16670694e-06)*t*t*t*e
        + (9.52692432e-07)*t*t*t*t*e
        + (-4.29223622e-02)*v*e
        + (1.96701861e-03)*t*v*e
        + (1.08799899e-05)*t*t*v*e
        + (2.63940916e-07)*t*t*t*v*e
        + (-2.52785852e-09)*t*t*t*t*v*e
        + (1.94960053e-02)*v*v*e
        + (-1.00361113e-03)*t*v*v*e
        + (-1.21206673e-05)*t*t*v*v*e
        + (-2.02156058e-07)*t*t*t*v*v*e
        + (-3.36514630e-05)*v*v*v*e
        + (1.35908359e-05)*t*v*v*v*e
        + (4.17032620e-07)*t*t*v*v*v*e
        + (2.17032620e-08)*v*v*v*v*e

        + (0.00174848160)*d
        + (0.00020711500)*t*d
        + (-0.00000228189)*t*t*d
        + (-0.00000011407)*t*t*t*d
        + (0.00000000425)*t*t*t*t*d
        + (-0.00001282736)*v*d
        + (-0.00000271153)*t*v*d
        + (0.00000002018)*t*t*v*d
        + (0.00000000044)*t*t*t*v*d
        + (0.00000001334)*v*v*d
        + (0.00000000127)*t*v*v*d
        + (-0.00000000005)*t*t*v*v*d

        + (-0.00009261467)*e*d
        + (0.00000100686)*t*e*d
        + (0.00000002380)*v*e*d
        + (-0.00000000012)*t*v*e*d
        + (0.00000000011)*v*v*e*d

        + (-0.00000009822)*d*d
        + (0.00000000119)*t*d*d
        + (0.00000000003)*v*d*d
    )

    utci[mask] = utci_m
    return utci

In [ ]:

# simple function to map
def plot_heatmap(data, title, cmap="viridis", mask_nan=True):
    plt.figure(figsize=(8, 6))
    arr = np.asarray(data)
    if mask_nan:
        arr = np.ma.masked_invalid(arr)
    sns.heatmap(arr, cmap=cmap, square=True, cbar=True)
    plt.title(title)
    plt.xlabel("Column")
    plt.ylabel("Row")
    plt.tight_layout()
    plt.show()

In [ ]:

# Compute SVF & Shade

if __name__ == "__main__":
    H = ar_ndsm.copy().astype(np.float32)
    # Clamp negative values to 0 (measurement artifacts)
    H[H < 0] = 0.0

    svf, shade, sun_az, sun_el = compute_svf_and_shade(
        H,
        LAT, LON,
        SIMULATION_DATETIME,
        tz_offset_hours=TZ_OFFSET,
        cellsize=CELLSIZE,
        n_azimuth=N_AZIMUTH,
        max_distance=MAX_DISTANCE_M,
        ground_threshold=GROUND_THRESHOLD_M,
    )

    print(f"Sun azimuth: {sun_az:.1f}°  Sun elevation: {sun_el:.1f}°")
    print(f"SVF shape: {svf.shape}  Shade shape: {shade.shape}")
    print(f"SVF finite pixels: {np.isfinite(svf).sum()} / {svf.size}")
    valid_svf = np.argwhere(np.isfinite(svf))
    if valid_svf.size > 0:
        r, c = valid_svf[0]
        print(f"  SVF at ({r}, {c}): {svf[r, c]:.4f}")
    else:
        print("  ⚠ All SVF values are NaN — check ground_threshold.")


In [ ]:
# plt.imshow(ar)

In [ ]:

# Compute MRT using auto-fetched weather


# Surface properties
albedo = np.full_like(svf, 0.20, dtype=np.float64)

# Solar geometry
azimuth, elevation = sun_position(LAT, LON, SIMULATION_DATETIME, TZ_OFFSET)

print(f"Weather inputs  →  Ta={Ta:.1f}°C, RH={RH:.0f}%, K_global={K_global:.0f} W/m², K_diffuse={K_diffuse:.0f} W/m²")
print(f"Solar position  →  Azimuth={azimuth:.1f}°, Elevation={elevation:.1f}°")

# Mean Radiant Temperature
mrt = compute_mrt_no_wind(
    svf=svf,
    shade=shade,
    Ta_C=Ta,
    RH_percent=RH,
    K_global=K_global,
    K_diffuse=K_diffuse,
    solar_elevation_deg=elevation,
    albedo=albedo
)

plot_heatmap(mrt, f"Mean Radiant Temperature (°C) — {SIMULATION_DATETIME}")
plot_heatmap(svf, "Sky View Factor (SVF)")
plot_heatmap(shade.astype(int), "Shade (1=shaded, 0=sunlit)", cmap="gray_r", mask_nan=False)


In [ ]:
plot_heatmap(mrt, "Mean Radiant Temperature (°C)")

In [ ]:
# ----------------------------
# Compute UTCI
# ----------------------------
wind = np.full_like(mrt, WIND_SPEED_MPS, dtype=np.float64)

utci = compute_utci_from_mrt(
    Ta_C=Ta,
    RH_percent=RH,
    wind_mps=wind,
    mrt_C=mrt,
)

plot_heatmap(utci, f"UTCI (°C) — {SIMULATION_DATETIME}")
print(f"UTCI range: {np.nanmin(utci):.1f} to {np.nanmax(utci):.1f} °C")


In [ ]:
np.nanmax(mrt)

In [ ]:
H

In [ ]:
#animation

from datetime import datetime

ANIMATION_DATE = datetime(2025, 7, 1)  # ← Just set the date (year, month, day)

# GIF settings
GIF_FPS = 2             # frames per second (3 = ~8 sec loop)
GIF_DPI = 100           # resolution

print(f"Will animate full day: {ANIMATION_DATE.strftime('%Y-%m-%d')}")
print(f"Using coordinates: {LAT:.5f}°N, {LON:.5f}°E")


In [ ]:
#weather and compute frames

import requests
from datetime import date

# ── Fetch full-day weather in one API call ───────────────────
date_str = ANIMATION_DATE.strftime('%Y-%m-%d')
days_ago = (date.today() - ANIMATION_DATE.date()).days

if days_ago > 5:
    base_url = 'https://archive-api.open-meteo.com/v1/archive'
else:
    base_url = 'https://api.open-meteo.com/v1/forecast'

params = {
    'latitude': LAT,
    'longitude': LON,
    'start_date': date_str,
    'end_date': date_str,
    'hourly': ','.join([
        'temperature_2m',
        'relative_humidity_2m',
        'shortwave_radiation',
        'diffuse_radiation',
        'wind_speed_10m',
    ]),
    'timezone': 'auto',
}

print(f"Fetching 24-hour weather for {date_str} ...")
resp = requests.get(base_url, params=params, timeout=15)
resp.raise_for_status()
wx_day = resp.json()['hourly']
print(f"  Got {len(wx_day['temperature_2m'])} hourly records.")

# ── Prepare height array (same as single-frame computation) ──
H_anim = ar_ndsm.copy().astype(np.float32)
H_anim[H_anim < 0] = 0.0

# ── SVF is computed once (geometry doesn't change with time) ─
print("\nComputing SVF (once — geometry only) ...")
# Use noon just to get a valid sun position for the SVF call,
# but SVF itself doesn't depend on time.
noon_dt = datetime(ANIMATION_DATE.year, ANIMATION_DATE.month, ANIMATION_DATE.day, 12, 0, 0)
svf_anim, _, _, _ = compute_svf_and_shade(
    H_anim, LAT, LON, noon_dt,
    tz_offset_hours=TZ_OFFSET,
    cellsize=CELLSIZE,
    n_azimuth=N_AZIMUTH,
    max_distance=MAX_DISTANCE_M,
    ground_threshold=GROUND_THRESHOLD_M,
)
print(f"  SVF done. Finite pixels: {np.isfinite(svf_anim).sum()} / {svf_anim.size}")

# ── Loop over 24 hours: compute shade + MRT + UTCI each hour ─
hourly_shade = []
hourly_mrt = []
hourly_utci = []
hourly_sun_el = []
hourly_labels = []

albedo_anim = np.full_like(svf_anim, 0.20, dtype=np.float64)

print("\nComputing hourly shade, MRT, and UTCI ...")
for hr in range(24):
    dt_hr = datetime(ANIMATION_DATE.year, ANIMATION_DATE.month, ANIMATION_DATE.day, hr, 0, 0)

    # Weather for this hour
    ta_hr = wx_day['temperature_2m'][hr]
    rh_hr = wx_day['relative_humidity_2m'][hr]
    kg_hr = wx_day['shortwave_radiation'][hr] or 0.0
    kd_hr = wx_day['diffuse_radiation'][hr] or 0.0
    ws_hr = wx_day['wind_speed_10m'][hr] or 2.0

    # Sun position & shade (recompute — shade depends on sun angle)
    _, shade_hr, sun_az_hr, sun_el_hr = compute_svf_and_shade(
        H_anim, LAT, LON, dt_hr,
        tz_offset_hours=TZ_OFFSET,
        cellsize=CELLSIZE,
        n_azimuth=N_AZIMUTH,
        max_distance=MAX_DISTANCE_M,
        ground_threshold=GROUND_THRESHOLD_M,
    )

    # MRT
    _, elevation_hr = sun_position(LAT, LON, dt_hr, TZ_OFFSET)
    mrt_hr = compute_mrt_no_wind(
        svf=svf_anim,
        shade=shade_hr,
        Ta_C=ta_hr,
        RH_percent=rh_hr,
        K_global=kg_hr,
        K_diffuse=kd_hr,
        solar_elevation_deg=elevation_hr,
        albedo=albedo_anim,
    )

    # UTCI
    wind_hr = np.full_like(mrt_hr, ws_hr, dtype=np.float64)
    utci_hr = compute_utci_from_mrt(
        Ta_C=ta_hr,
        RH_percent=rh_hr,
        wind_mps=wind_hr,
        mrt_C=mrt_hr,
    )

    hourly_shade.append(shade_hr)
    hourly_mrt.append(mrt_hr)
    hourly_utci.append(utci_hr)
    hourly_sun_el.append(elevation_hr)
    hourly_labels.append(dt_hr.strftime('%H:%M'))

    status = '☀' if elevation_hr > 0 else '🌙'
    print(f"  {dt_hr.strftime('%H:%M')} {status}  "
          f"sun_el={elevation_hr:+6.1f}°  "
          f"Ta={ta_hr:.1f}°C  RH={rh_hr:.0f}%  "
          f"K={kg_hr:.0f} W/m²")

print("\nDone! 24 hourly frames computed.")


In [ ]:
#automatic gif creation

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
import io
import os

# Output directory (Google Drive or local)
GIF_OUTPUT_DIR = '/content/drive/MyDrive/Codes-Urban Heat/Heat_Imaging_Files/'
os.makedirs(GIF_OUTPUT_DIR, exist_ok=True)

date_tag = ANIMATION_DATE.strftime('%Y%m%d')


def frames_to_gif(frames_data, title_template, filepath,
                   cmap='viridis', vmin=None, vmax=None,
                   mask_nan=True, fps=3, dpi=100):
    """
    Render a list of 2D arrays as a GIF.

    Parameters
    ----------
    frames_data : list of 2D arrays
    title_template : str with {hour} and optionally {date} placeholders
    filepath : output .gif path
    cmap : colormap name
    vmin, vmax : color scale bounds (auto if None)
    mask_nan : mask NaN pixels
    fps : frames per second
    dpi : image resolution
    """
    # Auto-detect color range across all frames
    if vmin is None:
        vmin = min(np.nanmin(f) for f in frames_data)
    if vmax is None:
        vmax = max(np.nanmax(f) for f in frames_data)

    pil_frames = []
    for i, frame in enumerate(frames_data):
        fig, ax = plt.subplots(figsize=(8, 6))
        arr = np.asarray(frame, dtype=float)
        if mask_nan:
            arr = np.ma.masked_invalid(arr)

        im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax,
                       aspect='equal', interpolation='nearest')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        title = title_template.format(
            hour=hourly_labels[i],
            date=ANIMATION_DATE.strftime('%Y-%m-%d'),
        )
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Column')
        ax.set_ylabel('Row')
        fig.tight_layout()

        # Render to PIL image
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        pil_frames.append(Image.open(buf).copy())

    # Save GIF
    duration_ms = int(1000 / fps)
    pil_frames[0].save(
        filepath,
        save_all=True,
        append_images=pil_frames[1:],
        duration=duration_ms,
        loop=0,
    )
    print(f"  Saved: {filepath}  ({len(pil_frames)} frames, {fps} fps)")


# ── 1. Shade GIF ────────────────────────────────────────────
print("Generating Shade GIF ...")
shade_frames = [s.astype(int) for s in hourly_shade]
frames_to_gif(
    shade_frames,
    title_template='Shade — {date} {hour}  (1=shaded, 0=sunlit)',
    filepath=os.path.join(GIF_OUTPUT_DIR, f'shade_{date_tag}.gif'),
    cmap='gray_r', vmin=0, vmax=1,
    mask_nan=False, fps=GIF_FPS, dpi=GIF_DPI,
)

# ── 2. MRT GIF ──────────────────────────────────────────────
print("Generating MRT GIF ...")
frames_to_gif(
    hourly_mrt,
    title_template='Mean Radiant Temperature (°C) — {date} {hour}',
    filepath=os.path.join(GIF_OUTPUT_DIR, f'mrt_{date_tag}.gif'),
    cmap='inferno', fps=GIF_FPS, dpi=GIF_DPI,
)

# ── 3. UTCI GIF ─────────────────────────────────────────────
print("Generating UTCI GIF ...")
frames_to_gif(
    hourly_utci,
    title_template='UTCI (°C) — {date} {hour}',
    filepath=os.path.join(GIF_OUTPUT_DIR, f'utci_{date_tag}.gif'),
    cmap='RdYlBu_r', fps=GIF_FPS, dpi=GIF_DPI,
)

# ── 4. SVF static plot (doesn't change hourly) ──────────────
print("\nSVF is time-independent — saving as a single image.")
fig, ax = plt.subplots(figsize=(8, 6))
svf_masked = np.ma.masked_invalid(svf_anim)
im = ax.imshow(svf_masked, cmap='viridis', vmin=0, vmax=1,
               aspect='equal', interpolation='nearest')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f'Sky View Factor — {ANIMATION_DATE.strftime("%Y-%m-%d")}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Column')
ax.set_ylabel('Row')
fig.tight_layout()
svf_path = os.path.join(GIF_OUTPUT_DIR, f'svf_{date_tag}.png')
fig.savefig(svf_path, dpi=GIF_DPI, bbox_inches='tight')
plt.show()
print(f"  Saved: {svf_path}")

print(f"\n✅ All outputs saved to: {GIF_OUTPUT_DIR}")


In [ ]:
#gif web display

from IPython.display import Image as IPImage, display, HTML

print("=" * 50)
print("SHADE")
print("=" * 50)
display(IPImage(filename=os.path.join(GIF_OUTPUT_DIR, f'shade_{date_tag}.gif')))

print("\n" + "=" * 50)
print("MEAN RADIANT TEMPERATURE")
print("=" * 50)
display(IPImage(filename=os.path.join(GIF_OUTPUT_DIR, f'mrt_{date_tag}.gif')))

print("\n" + "=" * 50)
print("UTCI")
print("=" * 50)
display(IPImage(filename=os.path.join(GIF_OUTPUT_DIR, f'utci_{date_tag}.gif')))
